## IMPORT LIBRARY

In [1]:
%pip install folium


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import json
import re
import folium

## IMPORT DATA

In [3]:
# 1. Baca data CSV
csv_file = "data_tpt_jateng.csv"
df = pd.read_csv(csv_file, on_bad_lines='skip')

kolom_wilayah = df.columns[0]
kolom_tpt = df.columns[1]
daftar_wilayah_csv = df[kolom_wilayah].dropna().unique().tolist()

# 2. Baca file GeoJSON
geojson_file = "peta_jateng.geojson"
with open(geojson_file, 'r', encoding='utf-8') as f:
    geojson_data = json.load(f)


## CLEANING DATA

In [4]:

# PENTING: Filter fitur GeoJSON untuk membuang data yang cacat (tidak punya properti/WADMKK)
geojson_data['features'] = [
    f for f in geojson_data['features']
    if f.get('properties') is not None and f['properties'].get('WADMKK') is not None
]

# 3. Fungsi pencocokan cerdas
def cocokkan_nama(nama_geojson, daftar_csv):
    nama_bersih = str(nama_geojson).strip().upper()
    for nama_csv in daftar_csv:
        if nama_bersih == str(nama_csv).strip().upper():
            return nama_csv

    nama_kunci_geo = re.sub(r'^(KABUPATEN|KOTA)\s+', '', nama_bersih)
    for nama_csv in daftar_csv:
        nama_kunci_csv = re.sub(r'^(KABUPATEN|KOTA)\s+', '', str(nama_csv).strip().upper())
        if nama_kunci_geo == nama_kunci_csv:
            is_kota_geo = 'KOTA' in nama_bersih
            is_kota_csv = 'KOTA' in str(nama_csv).strip().upper()
            if is_kota_geo == is_kota_csv:
                return nama_csv
    return nama_geojson

# 4. Sinkronisasi nama dan nilai TPT
for feature in geojson_data['features']:
    nama_lama = feature['properties']['WADMKK']
    nama_baru = cocokkan_nama(nama_lama, daftar_wilayah_csv)

    feature['properties']['WADMKK'] = nama_baru

    baris_data = df[df[kolom_wilayah] == nama_baru]
    if not baris_data.empty:
        feature['properties']['TPT'] = float(baris_data.iloc[0][kolom_tpt])
    else:
        feature['properties']['TPT'] = None

## MEMBUAT PETA JAWA TENGAH

In [5]:


# 5. Membuat Peta Interaktif
peta = folium.Map(location=[-7.150975, 110.140259], zoom_start=8, tiles=None)

# Pastikan key_on ditulis dengan awalan 'feature.'
folium.Choropleth(
    geo_data=geojson_data,
    name='Choropleth TPT',
    data=df,
    columns=[kolom_wilayah, kolom_tpt],
    key_on='feature.properties.WADMKK',
    fill_color='YlOrRd', 
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Tingkat Pengangguran Terbuka (%)',
    reset=True
).add_to(peta)

# Layer interaktif untuk klik dan zoom
folium.GeoJson(
    geojson_data,
    name="Interaksi Wilayah",
    style_function=lambda x: {'fillColor': 'transparent', 'color': 'transparent'},
    highlight_function=lambda x: {'weight': 3, 'color': 'black', 'fillOpacity': 0.1},
    tooltip=folium.features.GeoJsonTooltip(
        fields=['WADMKK', 'TPT'],
        aliases=['Wilayah:', 'TPT (%):'],
        localize=True,
        sticky=False
    ),
    zoom_on_click=True
).add_to(peta)

folium.LayerControl().add_to(peta)
output_html = "peta_interaktif_jateng_fix.html"
peta.save(output_html)
print(f"Selesai! Buka file '{output_html}' di browser Anda.")

Selesai! Buka file 'peta_interaktif_jateng_fix.html' di browser Anda.
